# 🎯 Clase 5 — Regresión Logística y Scoring Crediticio

### Aplicaciones de Minería de Datos a Economía y Finanzas
**Maestría en Minería de Datos · UTN Facultad Regional Rosario · 2026**

*Dr. Darío Ezequiel Díaz · Jueves 7 de mayo de 2026*

---

## 📋 Objetivos de la práctica

Al finalizar este notebook, ustedes serán capaces de:

1. **Ajustar** un modelo de regresión logística sobre el German Credit Dataset usando dos enfoques complementarios:
   - `scikit-learn` para predicción
   - `statsmodels` para inferencia estadística
2. **Interpretar** los coeficientes del modelo a través de odds ratios e intervalos de confianza al 95%
3. **Construir** una scorecard operativa siguiendo la formulación de Siddiqi (PDO = 20, Score_target = 600, Odds_target = 50:1)
4. **Evaluar** la performance del modelo mediante AUC-ROC, KS y Gini
5. **Diagnosticar** los supuestos del modelo: linealidad en logits, multicolinealidad (VIF) y eventos por variable (EPV)

## 🗺️ Mapa del notebook

| Sección | Contenido |
|---------|-----------|
| 1 | Configuración del entorno |
| 2 | Carga del dataset German Credit |
| 3 | Exploración inicial |
| 4 | Preprocesamiento |
| 5 | Ajuste con scikit-learn |
| 6 | Ajuste con statsmodels e interpretación |
| 7 | Construcción de la scorecard |
| 8 | Métricas de evaluación: AUC, KS y Gini |
| 9 | Diagnóstico de supuestos |
| 10 | Síntesis y entregable |

---

> 💡 **Recomendación:** ejecuten las celdas en orden secuencial. Cada sección depende de los objetos definidos en las anteriores.


## 1. Configuración del entorno

Instalamos las librerías necesarias y fijamos la semilla aleatoria del curso (`RNG_SEED = 2026`) para garantizar reproducibilidad de resultados.

### Librerías utilizadas

- **`pandas`, `numpy`**: manipulación de datos y cálculo numérico
- **`matplotlib`, `seaborn`**: visualización
- **`scikit-learn`**: ajuste del modelo, métricas y partición train/test
- **`statsmodels`**: inferencia estadística (errores estándar, p-valores, IC)
- **`scipy.stats`**: KS de Kolmogorov-Smirnov


In [ ]:
# Instalación de librerías (silenciosa si ya están disponibles)
import subprocess
import sys

paquetes_requeridos = ['statsmodels', 'scikit-learn', 'pandas', 'numpy',
                       'matplotlib', 'seaborn', 'scipy']

for pkg in paquetes_requeridos:
    try:
        __import__(pkg.replace('-', '_'))
    except ImportError:
        print(f"📦 Instalando {pkg}...")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                              '--quiet', '--break-system-packages', pkg])

print("✅ Todas las librerías están disponibles")


In [ ]:
# Imports principales
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, roc_curve, confusion_matrix,
                             classification_report)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats

# Configuración global
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

RNG_SEED = 2026
np.random.seed(RNG_SEED)

# Paleta cromática Unidad 3 — Royal Sapphire & Gold
COLOR_INDIGO  = '#1E1B4B'  # primario: indigo profundo
COLOR_VIOLET  = '#6D28D9'  # secundario: violeta royal
COLOR_GOLD    = '#D97706'  # acento: ámbar dorado
COLOR_EMERALD = '#059669'  # good / aprobado
COLOR_CARMIN  = '#DC2626'  # bad / default
COLOR_GRAY    = '#64748B'  # texto secundario

PALETA_BG = {'good': COLOR_EMERALD, 'bad': COLOR_CARMIN}

# Configuración estética global de matplotlib
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'figure.dpi': 100,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.titlecolor': COLOR_INDIGO,
    'axes.labelsize': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 10,
    'legend.frameon': False,
})
sns.set_palette([COLOR_VIOLET, COLOR_GOLD, COLOR_EMERALD, COLOR_CARMIN])

print(f"🎯 RNG_SEED = {RNG_SEED}")
print(f"🎨 Paleta cromática Unidad 3 cargada")
print(f"📚 Librerías importadas correctamente")


## 2. Carga del German Credit Dataset

Cargamos el conjunto de datos desde **OpenML** mediante `sklearn.datasets.fetch_openml('credit-g')`. Este dataset, originado en investigación de Hofmann (Universidad de Hamburgo, 1994), contiene **1.000 solicitudes de crédito** con 20 variables explicativas y una variable respuesta binaria:

- `Y = 'good'`: el solicitante pagó regularmente (700 casos, 70 %)
- `Y = 'bad'`: el solicitante incumplió (300 casos, 30 %)

> ⚠️ **Convención del curso**: codificamos `Y = 1` para el evento adverso (default), siguiendo la práctica regulatoria de Basilea. Por tanto: `bad → 1`, `good → 0`.

### Estrategia de carga resiliente

Implementamos un *fallback* en cascada:
1. **Primer intento**: `fetch_openml('credit-g')` desde OpenML
2. **Segundo intento**: CSV local desde Google Drive (`/content/drive/MyDrive/datasets/german_credit.csv`)
3. **Tercer intento**: descarga directa desde el repositorio del curso


In [ ]:
def cargar_german_credit():
    '''
    Carga el German Credit Dataset con estrategia de fallback en tres niveles.
    Retorna un DataFrame con las 20 covariables y la variable respuesta 'class'.
    '''
    # Intento 1: OpenML
    try:
        print("⏳ Intentando cargar desde OpenML...")
        data = fetch_openml('credit-g', version=1, as_frame=True, parser='auto')
        df = data.frame
        print(f"✅ Dataset cargado desde OpenML: {df.shape[0]} filas × {df.shape[1]} columnas")
        return df
    except Exception as e:
        print(f"⚠️  OpenML no disponible: {str(e)[:80]}")

    # Intento 2: Google Drive (en Colab)
    try:
        ruta_drive = '/content/drive/MyDrive/datasets/german_credit.csv'
        df = pd.read_csv(ruta_drive)
        print(f"✅ Dataset cargado desde Drive: {df.shape[0]} × {df.shape[1]}")
        return df
    except Exception:
        pass

    # Intento 3: URL pública del repositorio
    try:
        url = 'https://www.openml.org/data/get_csv/31/dataset_31_credit-g.arff'
        df = pd.read_csv(url)
        print(f"✅ Dataset cargado desde URL pública: {df.shape[0]} × {df.shape[1]}")
        return df
    except Exception as e:
        raise RuntimeError(f"❌ No se pudo cargar el dataset por ninguna vía. Último error: {e}")


df = cargar_german_credit()
df.head()


In [ ]:
# Recodificación de la variable respuesta según convención regulatoria
# Y = 1 representa el evento adverso (default); Y = 0 el pago regular

df['Y'] = (df['class'] == 'bad').astype(int)

# Verificamos la distribución
print("📊 Distribución de la variable respuesta:")
print(df['Y'].value_counts())
print(f"\n   Tasa de default observada: {df['Y'].mean():.2%}")
print(f"   Tasa de pago regular     : {1 - df['Y'].mean():.2%}")


## 3. Exploración inicial

Antes de modelar, inspeccionamos la estructura del dataset: tipos de variables, presencia de faltantes y distribución de la variable respuesta por algunas covariables candidatas.

### 3.1. Estructura del dataset


In [ ]:
# Información estructural
print(f"📐 Dimensiones: {df.shape}")
print(f"📐 Faltantes totales: {df.isna().sum().sum()}")
print(f"\n🔍 Tipos de variables:")
print(df.dtypes.value_counts())


In [ ]:
# Resumen estadístico de las variables numéricas
df.select_dtypes(include=[np.number]).describe().round(2)


### 3.2. Distribución de la variable respuesta

Visualizamos la proporción de buenos (`Y = 0`) y malos (`Y = 1`) en el dataset. El desbalanceo del 70/30 es moderado y manejable sin técnicas de resampleo agresivas (las veremos en la Unidad 4).


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))

# Panel izquierdo: barras
counts = df['Y'].value_counts().sort_index()
labels = ['Pago regular (Y=0)', 'Default (Y=1)']
colores = [COLOR_EMERALD, COLOR_CARMIN]
ax[0].bar(labels, counts.values, color=colores, edgecolor='white', linewidth=2)
for i, v in enumerate(counts.values):
    ax[0].text(i, v + 15, f'{v}\n({v/len(df):.1%})',
               ha='center', fontweight='bold', color=COLOR_INDIGO)
ax[0].set_ylabel('Frecuencia absoluta')
ax[0].set_title('Distribución de la variable respuesta')
ax[0].set_ylim(0, max(counts) * 1.15)

# Panel derecho: anillo
ax[1].pie(counts.values, labels=labels, colors=colores, autopct='%1.1f%%',
          startangle=90, wedgeprops=dict(width=0.35, edgecolor='white'),
          textprops={'fontsize': 11, 'fontweight': 'bold'})
ax[1].set_title('Proporción de clases')

plt.tight_layout()
plt.show()


### 3.3. Variables candidatas: visualización exploratoria

Seleccionamos algunas variables prometedoras del dataset y exploramos su relación con la variable respuesta.


In [ ]:
# Visualización de relaciones entre covariables clave y la respuesta
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Panel 1: Duración del crédito
sns.kdeplot(data=df, x='duration', hue='Y', ax=axes[0, 0],
            palette=[COLOR_EMERALD, COLOR_CARMIN], fill=True, alpha=0.5)
axes[0, 0].set_title('Duración del crédito (meses)')
axes[0, 0].set_xlabel('Meses')
axes[0, 0].legend(['Default', 'Pago regular'])

# Panel 2: Monto del crédito (escala log para apreciar mejor)
sns.kdeplot(data=df, x='credit_amount', hue='Y', ax=axes[0, 1],
            palette=[COLOR_EMERALD, COLOR_CARMIN], fill=True, alpha=0.5,
            log_scale=True)
axes[0, 1].set_title('Monto del crédito (escala log)')
axes[0, 1].set_xlabel('Monto')
axes[0, 1].legend(['Default', 'Pago regular'])

# Panel 3: Edad
sns.kdeplot(data=df, x='age', hue='Y', ax=axes[1, 0],
            palette=[COLOR_EMERALD, COLOR_CARMIN], fill=True, alpha=0.5)
axes[1, 0].set_title('Edad del solicitante (años)')
axes[1, 0].set_xlabel('Edad')
axes[1, 0].legend(['Default', 'Pago regular'])

# Panel 4: Tasa de default por historial crediticio
tasa_hist = df.groupby('credit_history')['Y'].mean().sort_values(ascending=False)
bars = axes[1, 1].barh(range(len(tasa_hist)), tasa_hist.values, color=COLOR_VIOLET,
                       edgecolor='white', linewidth=1.5)
axes[1, 1].set_yticks(range(len(tasa_hist)))
axes[1, 1].set_yticklabels([s[:25] for s in tasa_hist.index], fontsize=9)
axes[1, 1].set_xlabel('Tasa de default')
axes[1, 1].set_title('Tasa de default por historial crediticio')
axes[1, 1].axvline(df['Y'].mean(), color=COLOR_GOLD, linestyle='--',
                   linewidth=2, label=f'Media global: {df["Y"].mean():.2%}')
axes[1, 1].legend(loc='lower right')
for i, v in enumerate(tasa_hist.values):
    axes[1, 1].text(v + 0.005, i, f'{v:.1%}', va='center', fontsize=9, color=COLOR_INDIGO)

plt.tight_layout()
plt.show()


### 📝 Interpretación exploratoria

**Lectura preliminar de las visualizaciones:**

- **Duración del crédito**: la distribución de duraciones para los defaults (rojo) está visiblemente desplazada hacia valores más altos respecto de los pagos regulares (verde). Coherente con la intuición: créditos más largos exponen más al solicitante a eventos adversos.

- **Monto del crédito**: en escala logarítmica, los defaults se concentran ligeramente hacia montos mayores. El efecto es menos marcado que el de la duración.

- **Edad**: distribución más concentrada hacia los 25-35 años en los defaults, mientras que los pagos regulares se distribuyen de manera más uniforme. La edad parece comportarse como factor protector.

- **Historial crediticio**: variable categórica con efecto fuerte. Los solicitantes con historial 'critical' o 'no credits/all paid' presentan tasas de default sustancialmente superiores a la media. Será una variable clave del modelo.


## 4. Preprocesamiento

Preparamos los datos para el ajuste del modelo siguiendo cuatro pasos:

1. **Selección de variables**: trabajaremos con un subconjunto de 8 variables (5 continuas y 3 categóricas) para que el modelo sea interpretable y la scorecard manejable.
2. **Codificación**: las variables categóricas se transforman mediante *one-hot encoding* dejando una categoría como referencia.
3. **Partición train/test**: 70/30 estratificado por la variable respuesta para preservar la proporción de clases.
4. **Estandarización**: las variables continuas se escalan a media 0 y desvío 1 (recomendable para la convergencia del solver `lbfgs` de sklearn).


In [ ]:
# Selección de variables del modelo
vars_continuas = ['duration', 'credit_amount', 'age',
                  'installment_commitment', 'residence_since']
vars_categoricas = ['credit_history', 'purpose', 'savings_status']

X_raw = df[vars_continuas + vars_categoricas].copy()
y = df['Y'].copy()

print(f"📋 Variables continuas    ({len(vars_continuas)}): {vars_continuas}")
print(f"📋 Variables categóricas  ({len(vars_categoricas)}): {vars_categoricas}")
print(f"📋 Total covariables: {len(vars_continuas) + len(vars_categoricas)}")


### 4.1. Diagnóstico de cardinalidad: prevención de separación perfecta

Antes de codificar las variables categóricas, debemos **inspeccionar la frecuencia de cada categoría** y la tasa de default por categoría. Las categorías con muy pocos casos pueden producir dos problemas serios:

1. **Estimadores inestables**: con 5 o 10 casos en una categoría, el coeficiente asociado tiene un error estándar enorme y la interpretación deja de tener sentido.
2. **Separación perfecta**: si todos los casos de una categoría rara son de la misma clase (todos buenos o todos malos), el algoritmo de máxima verosimilitud **no converge** y los coeficientes divergen a $\pm\infty$.

Este es uno de los problemas técnicos más frecuentes en *credit scoring* real. Lo abordamos preventivamente: **agruparemos en una categoría única `other_minor`** todas las categorías con menos de 30 casos.


In [ ]:
# Diagnóstico: frecuencia y tasa de default por categoría
UMBRAL_FRECUENCIA = 30  # mínimo número de casos por categoría

print("🔍 Diagnóstico de cardinalidad por variable categórica\n")
print(f"   Umbral de frecuencia mínima: {UMBRAL_FRECUENCIA} casos\n")

diagnosticos = {}
for var in vars_categoricas:
    tabla = pd.crosstab(df[var], df['Y'], margins=True, margins_name='Total')
    tabla.columns = ['Y=0 (good)', 'Y=1 (bad)', 'Total']
    tabla['Tasa default'] = (tabla['Y=1 (bad)'] / tabla['Total']).round(3)

    # Marcamos categorías problemáticas
    tabla['Diagnóstico'] = ''
    for cat in tabla.index[:-1]:  # excluimos el 'Total'
        n_total = tabla.loc[cat, 'Total']
        n_bad = tabla.loc[cat, 'Y=1 (bad)']
        n_good = tabla.loc[cat, 'Y=0 (good)']
        if n_total < UMBRAL_FRECUENCIA:
            if n_bad == 0 or n_good == 0:
                tabla.loc[cat, 'Diagnóstico'] = '❌ Separación + raro'
            else:
                tabla.loc[cat, 'Diagnóstico'] = '⚠️  Raro (n < 30)'
        elif n_bad == 0 or n_good == 0:
            tabla.loc[cat, 'Diagnóstico'] = '❌ Separación perfecta'

    diagnosticos[var] = tabla
    print(f"━━━ {var} ━━━")
    print(tabla.to_string())
    print()


In [ ]:
# Agrupamiento de categorías raras en 'other_minor'
# Esta operación previene la separación perfecta y estabiliza los estimadores

X_raw_agrupado = X_raw.copy()

print("🔧 Agrupando categorías con menos de 30 casos en 'other_minor':\n")

for var in vars_categoricas:
    freq = X_raw[var].value_counts()
    categorias_raras = freq[freq < UMBRAL_FRECUENCIA].index.tolist()

    if categorias_raras:
        print(f"   {var}:")
        for cat in categorias_raras:
            print(f"      • '{cat}': {freq[cat]} casos → agrupar")
        # Reemplazamos las raras por 'other_minor'
        X_raw_agrupado[var] = X_raw[var].astype(str).where(
            ~X_raw[var].astype(str).isin(categorias_raras),
            'other_minor'
        )
        # Verificamos el resultado
        nueva_freq = X_raw_agrupado[var].value_counts()
        if 'other_minor' in nueva_freq.index:
            print(f"      → Nueva categoría 'other_minor' con {nueva_freq['other_minor']} casos\n")
    else:
        print(f"   {var}: ninguna categoría rara, no requiere agrupamiento.\n")

# Reemplazamos X_raw por la versión agrupada
X_raw = X_raw_agrupado
print("✅ Variables categóricas agrupadas. Continuamos con la versión limpia.")


In [ ]:
# Verificación EPV (Events Per Variable) — regla de Peduzzi (1996)
n_eventos = y.sum()
# Cardinalidad efectiva tras agrupamiento (usa X_raw ya limpio)
n_categorias_total = sum(X_raw[v].nunique() - 1 for v in vars_categoricas)  # -1 por dummy
n_variables_total = len(vars_continuas) + n_categorias_total

EPV = n_eventos / n_variables_total

print(f"🔍 Diagnóstico EPV (Events Per Variable)")
print(f"   Eventos (Y=1)             : {n_eventos}")
print(f"   Variables (con dummies)   : {n_variables_total}")
print(f"   EPV                       : {EPV:.1f}")
print(f"\n   Regla de Peduzzi: EPV ≥ 10 recomendado")
if EPV >= 10:
    print(f"   ✅ EPV = {EPV:.1f} cumple holgadamente la regla.")
elif EPV >= 5:
    print(f"   ⚠️  EPV = {EPV:.1f} entre 5 y 10: aceptable con regularización.")
else:
    print(f"   ❌ EPV = {EPV:.1f} < 5: alto riesgo de sobreajuste.")


In [ ]:
# Codificación one-hot de las variables categóricas
# drop='first' deja la primera categoría como referencia
encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')

X_cat = encoder.fit_transform(X_raw[vars_categoricas])
cat_columns = encoder.get_feature_names_out(vars_categoricas)

X_encoded = pd.DataFrame(X_cat, columns=cat_columns, index=X_raw.index)
X = pd.concat([X_raw[vars_continuas], X_encoded], axis=1)

print(f"📐 Matriz de diseño X: {X.shape[0]} filas × {X.shape[1]} columnas")
print(f"\n📋 Primeras columnas:")
print(X.columns.tolist()[:8])


In [ ]:
# Partición train/test estratificada por Y
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y,
    random_state=RNG_SEED
)

print(f"🎓 Train: {X_train.shape[0]} filas — tasa default: {y_train.mean():.2%}")
print(f"🧪 Test : {X_test.shape[0]} filas — tasa default: {y_test.mean():.2%}")


In [ ]:
# Estandarización de variables continuas (ajustar SOLO con train)
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[vars_continuas] = scaler.fit_transform(X_train[vars_continuas])
X_test_scaled[vars_continuas] = scaler.transform(X_test[vars_continuas])

print("✅ Variables continuas estandarizadas (μ=0, σ=1) en train y aplicadas a test")
print(f"\n   Verificación en train (debe ser ≈ 0 y ≈ 1):")
print(f"   Media: {X_train_scaled[vars_continuas].mean().mean():.6f}")
print(f"   Desvío: {X_train_scaled[vars_continuas].std().mean():.6f}")


## 5. Ajuste con scikit-learn

`LogisticRegression` es la clase central de sklearn para regresión logística. Está orientada a **producción y pipelines predictivos**.

### ⚠️ Advertencia silenciosa

A diferencia de `glm()` de R, **sklearn aplica regularización L2 por defecto** con `C=1.0`. Si queremos comparar coeficientes con statsmodels o R, debemos especificar `penalty=None` para obtener máxima verosimilitud pura.

Vamos a ajustar **dos modelos en paralelo**:
- **Modelo A**: con regularización L2 (default de sklearn)
- **Modelo B**: sin regularización (equivalente a MV pura)

Y compararemos sus coeficientes.


In [ ]:
# Modelo A: con regularización L2 (default de sklearn)
modelo_sklearn_l2 = LogisticRegression(
    penalty='l2',
    C=1.0,
    solver='lbfgs',
    max_iter=1000,
    random_state=RNG_SEED
)
modelo_sklearn_l2.fit(X_train_scaled, y_train)

# Modelo B: sin regularización (máxima verosimilitud pura)
modelo_sklearn_mv = LogisticRegression(
    penalty=None,
    solver='lbfgs',
    max_iter=1000,
    random_state=RNG_SEED
)
modelo_sklearn_mv.fit(X_train_scaled, y_train)

print("✅ Ambos modelos ajustados con scikit-learn")
print(f"   Modelo A (L2):  C={modelo_sklearn_l2.C}")
print(f"   Modelo B (MV):  sin regularización")


In [ ]:
# Comparación de coeficientes entre ambos modelos
comparacion = pd.DataFrame({
    'Variable': X_train.columns,
    'β (L2, C=1)': modelo_sklearn_l2.coef_[0].round(4),
    'β (MV pura)': modelo_sklearn_mv.coef_[0].round(4),
})
comparacion['Diferencia'] = (comparacion['β (L2, C=1)'] - comparacion['β (MV pura)']).round(4)
comparacion['|Δ| %'] = (np.abs(comparacion['Diferencia']) /
                       np.abs(comparacion['β (MV pura)']) * 100).round(1)

# Mostramos las primeras 10 filas
comparacion.head(10).style.background_gradient(
    subset=['|Δ| %'], cmap='RdYlGn_r'
).format({'|Δ| %': '{:.1f}%'})


### 📝 Interpretación: regularización L2 vs MV pura

La columna `|Δ| %` muestra el porcentaje de cambio entre los coeficientes regularizados y los de máxima verosimilitud pura. Como esperamos:

- **Diferencias pequeñas** (típicamente < 10 %) confirman que con tamaño muestral adecuado (n=700), la regularización L2 con C=1.0 introduce un sesgo modesto.
- En muestras más pequeñas o con multicolinealidad severa, las diferencias serían sustancialmente mayores.
- **Para el resto del notebook**, trabajaremos con el modelo de máxima verosimilitud pura, que es lo que statsmodels y R por defecto ajustan.


## 6. Ajuste con statsmodels e interpretación

`statsmodels` ofrece la salida clásica de inferencia estadística: errores estándar, p-valores e intervalos de confianza. Es la herramienta de elección para la **fase de análisis e interpretación** de coeficientes.

### Detalle importante

A diferencia de sklearn, statsmodels **no agrega el intercepto automáticamente**. Debemos hacerlo manualmente con `sm.add_constant()`.


In [ ]:
# Agregamos la constante para el intercepto
X_train_const = sm.add_constant(X_train_scaled, has_constant='add')
X_test_const  = sm.add_constant(X_test_scaled,  has_constant='add')

# Ajuste por máxima verosimilitud (Newton-Raphson)
logit_sm = sm.Logit(y_train, X_train_const).fit(disp=False, maxiter=100)

# ─── Verificación crítica de convergencia ───
convergencia = logit_sm.mle_retvals.get('converged', False)
n_iter       = logit_sm.mle_retvals.get('iterations', None)

print("🔍 Diagnóstico del ajuste por máxima verosimilitud:\n")
if convergencia:
    print(f"   ✅ Convergencia alcanzada en {n_iter} iteraciones.")
    print(f"   ✅ Log-verosimilitud final: {logit_sm.llf:.2f}")
    print(f"   ✅ Pseudo R² (McFadden):    {logit_sm.prsquared:.4f}")
else:
    print(f"   ❌ El algoritmo NO convergió tras {n_iter} iteraciones.")
    print(f"   ⚠️  Causa probable: separación perfecta en alguna covariable.")
    print(f"   📌 Inspeccionar coeficientes con |β| > 5 o SE > 10.")

# Detección de coeficientes patológicos
beta_max   = logit_sm.params.abs().max()
se_max     = logit_sm.bse.max()
var_extrema = logit_sm.params.abs().idxmax()

if beta_max > 5 or se_max > 10:
    print(f"\n   ⚠️  Variable con coeficiente extremo: '{var_extrema}'")
    print(f"      β̂ = {logit_sm.params[var_extrema]:.2f}, "
          f"SE = {logit_sm.bse[var_extrema]:.2f}")
    print(f"      → Esta variable probablemente debe agruparse o eliminarse.")
else:
    print(f"\n   ✅ Coeficientes en rango razonable (|β̂| max = {beta_max:.2f}, "
          f"SE max = {se_max:.2f}).")


**Resumen completo del modelo:**

In [ ]:
# Resumen estadístico completo (formato clásico)
print(logit_sm.summary().as_text()[:2500])


### 6.1. Tabla de coeficientes con odds ratio e intervalos de confianza

Construimos la tabla profesional que normalmente se reporta en un informe regulatorio: coeficiente estimado, error estándar, estadístico de Wald, p-valor, odds ratio y su intervalo de confianza al 95 %.


In [ ]:
# Construcción de la tabla profesional de resultados
params = logit_sm.params
bse    = logit_sm.bse
zvalor = logit_sm.tvalues
pvalor = logit_sm.pvalues
ic     = logit_sm.conf_int(alpha=0.05)

tabla_resultados = pd.DataFrame({
    'β̂':         params.round(4),
    'SE(β̂)':     bse.round(4),
    'z':         zvalor.round(3),
    'p-valor':   pvalor.round(4),
    'OR = exp(β̂)':       np.exp(params).round(3),
    'IC 95 % OR (inf)':  np.exp(ic[0]).round(3),
    'IC 95 % OR (sup)':  np.exp(ic[1]).round(3),
})

# Marcamos significancia
def marcar_sig(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    if p < 0.10:  return '.'
    return ''
tabla_resultados['Sig.'] = pvalor.apply(marcar_sig)

print("📊 Tabla profesional de coeficientes")
print("   Códigos de significancia: *** p<0.001, ** p<0.01, * p<0.05, . p<0.10\n")
tabla_resultados


### 6.2. Visualización del odds ratio con intervalos de confianza

Un **forest plot** es la representación estándar para visualizar odds ratios. Las variables se ordenan por magnitud absoluta del efecto y se muestra el IC 95 % como barra horizontal.


In [ ]:
# Forest plot de odds ratios (excluimos el intercepto)
tabla_plot = tabla_resultados.drop('const').copy()
tabla_plot = tabla_plot.sort_values('OR = exp(β̂)', ascending=True)

# Color según signo del coeficiente (OR > 1: rojo riesgo; OR < 1: verde protector)
colores = [COLOR_CARMIN if or_ > 1 else COLOR_EMERALD
           for or_ in tabla_plot['OR = exp(β̂)']]

fig, ax = plt.subplots(figsize=(11, 8))

# IC 95 % como barras horizontales
y_pos = np.arange(len(tabla_plot))
ax.hlines(y_pos, tabla_plot['IC 95 % OR (inf)'], tabla_plot['IC 95 % OR (sup)'],
          colors=colores, linewidth=2, alpha=0.6)

# Puntos para el OR estimado
ax.scatter(tabla_plot['OR = exp(β̂)'], y_pos, color=colores, s=120,
           edgecolor='white', linewidth=2, zorder=3)

# Línea de referencia OR = 1 (sin efecto)
ax.axvline(1, color=COLOR_INDIGO, linestyle='--', linewidth=1.5, alpha=0.7)
ax.text(1.02, len(tabla_plot) - 1, 'OR = 1\n(sin efecto)',
        fontsize=9, color=COLOR_INDIGO, va='top')

# Etiquetas
ax.set_yticks(y_pos)
ax.set_yticklabels([nombre[:35] for nombre in tabla_plot.index], fontsize=9)
ax.set_xlabel('Odds Ratio (escala lineal)')
ax.set_title('Odds ratios estimados con IC 95 %\nModelo: Regresión Logística — German Credit')
ax.grid(axis='x', alpha=0.3, linestyle=':')

# Anotación de la dirección del efecto
ax.text(0.02, 0.02, '← Reduce riesgo de default', transform=ax.transAxes,
        fontsize=10, color=COLOR_EMERALD, fontweight='bold')
ax.text(0.98, 0.02, 'Incrementa riesgo →', transform=ax.transAxes,
        fontsize=10, color=COLOR_CARMIN, fontweight='bold', ha='right')

plt.tight_layout()
plt.show()


### 📝 Lectura económica de los principales coeficientes

A partir del forest plot y la tabla, podemos elaborar la **lectura económica** de los efectos más relevantes:

- **Duración del crédito** (continua, estandarizada): OR > 1 indica que créditos más largos incrementan los odds de default. Es la variable temporal clásica de riesgo.

- **Monto del crédito** (continua, estandarizada): OR > 1 confirma que montos mayores se asocian a mayor riesgo, aunque el efecto suele ser menor que el de la duración.

- **Edad** (continua, estandarizada): OR < 1 confirma que la edad es **factor protector** — la madurez financiera reduce el riesgo de incumplimiento.

- **Historial crediticio**: las categorías 'critical' y 'no credits/all paid' presentan OR > 1 respecto de la categoría de referencia. El historial pasado predice el comportamiento futuro.

- **Propósito**: ciertos propósitos (radio/tv, education, business) muestran OR específicos que reflejan diferencias estructurales en el comportamiento de pago por tipo de gasto.

> 💡 La interpretación rigurosa requiere atender no solo al signo del coeficiente sino también a su **significancia estadística** (columna `Sig.`). Variables no significativas no deberían incorporarse al discurso interpretativo definitivo.


## 7. Construcción de la scorecard

Aplicamos la formulación de Siddiqi (2017) para transformar la probabilidad de default en una escala de puntaje operativa.

### Parámetros de la scorecard (configuración estándar de la industria)

| Parámetro | Valor | Descripción |
|-----------|-------|-------------|
| **PDO** | 20 | Points to Double the Odds: 20 puntos duplican los odds |
| **Score_target** | 600 | Puntaje de anclaje |
| **Odds_target** | 50:1 (good:bad) | Odds asociados al puntaje de anclaje |

### Fórmulas derivadas

$$\text{Factor} = \frac{\text{PDO}}{\ln 2} \qquad \text{Offset} = \text{Score\_target} - \text{Factor} \cdot \ln(\text{Odds\_target})$$

$$\text{Score} = \text{Offset} + \text{Factor} \cdot \ln(\text{Odds}_{\text{good:bad}})$$

> 🔑 **Convención de signos**: en la scorecard industrial, los odds se reportan como `good:bad` (favorable a desfavorable), por lo que **a mayor score, menor riesgo**.


In [ ]:
# Parámetros de la scorecard
PDO          = 20
SCORE_TARGET = 600
ODDS_TARGET  = 50  # good:bad ratio

# Derivación de Factor y Offset
FACTOR = PDO / np.log(2)
OFFSET = SCORE_TARGET - FACTOR * np.log(ODDS_TARGET)

print(f"📐 Parámetros derivados de la scorecard")
print(f"   PDO          = {PDO}")
print(f"   Score_target = {SCORE_TARGET}")
print(f"   Odds_target  = {ODDS_TARGET}:1 (good:bad)")
print(f"")
print(f"   Factor = PDO / ln(2)                          = {FACTOR:.4f}")
print(f"   Offset = Score_target − Factor · ln(Odds_target) = {OFFSET:.4f}")
print(f"")
print(f"   Fórmula: Score = {OFFSET:.2f} + {FACTOR:.2f} · ln(Odds_good_bad)")


In [ ]:
def prob_a_score(p, factor=FACTOR, offset=OFFSET):
    '''
    Convierte la probabilidad de default en score crediticio.

    A mayor score, menor riesgo (convención industrial).
    Los odds se calculan como good:bad = (1-p)/p.
    '''
    odds_good_bad = (1 - p) / p
    score = offset + factor * np.log(odds_good_bad)
    return score


# Generamos predicciones y scores para el conjunto de testeo
p_test = logit_sm.predict(X_test_const)
scores_test = prob_a_score(p_test)

# Resumen estadístico del score
resumen_score = pd.DataFrame({
    'Estadístico': ['Mínimo', 'Q1', 'Mediana', 'Media', 'Q3', 'Máximo',
                    'Desvío estándar'],
    'Score (test)': [
        scores_test.min(), scores_test.quantile(0.25),
        scores_test.median(), scores_test.mean(),
        scores_test.quantile(0.75), scores_test.max(),
        scores_test.std()
    ]
})
resumen_score['Score (test)'] = resumen_score['Score (test)'].round(1)
resumen_score


In [ ]:
# Verificación numérica: PDO debe efectivamente duplicar los odds good:bad
# Construimos dos pares (p_bad, odds_good_bad) donde odds difiere por factor 2

# Caso A: p_bad = 0.10  →  odds_good_bad = 0.90/0.10 = 9
# Caso B: p_bad ≈ 0.0526 → odds_good_bad = 0.947/0.0526 = 18 (= 9 × 2)

p_A = 0.10
odds_A = (1 - p_A) / p_A          # 9.0
odds_B = odds_A * 2                # 18.0
p_B = 1 / (1 + odds_B)             # ≈ 0.0526

s_A = prob_a_score(p_A)
s_B = prob_a_score(p_B)

diferencia = s_B - s_A   # debe ser exactamente PDO

print(f"🔍 Verificación PDO (Points to Double the Odds):")
print(f"")
print(f"   Caso A: p_bad = {p_A:.4f}  →  odds_good_bad = {odds_A:.2f}")
print(f"           Score = {s_A:.2f}")
print(f"")
print(f"   Caso B: p_bad = {p_B:.4f}  →  odds_good_bad = {odds_B:.2f}  (= odds_A × 2)")
print(f"           Score = {s_B:.2f}")
print(f"")
print(f"   ΔScore = {diferencia:.2f} puntos")
print(f"   PDO esperado: {PDO} puntos cuando los odds se duplican")
if abs(diferencia - PDO) < 0.01:
    print(f"   ✅ La fórmula opera correctamente (precisión perfecta).")
else:
    print(f"   ⚠️  Verificar derivación: diferencia = {abs(diferencia - PDO):.4f}")


### 7.1. Distribución de scores por clase

Visualizamos cómo se distribuyen los scores en el conjunto de testeo, separando buenos y malos. **Una buena scorecard separa visualmente ambas distribuciones**: los pagos regulares (verde) deben concentrarse en scores altos y los defaults (rojo) en scores bajos.


In [ ]:
# Distribución del score por clase
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel izquierdo: KDE superpuestas
df_score = pd.DataFrame({'Score': scores_test, 'Y': y_test.values})

for clase, color, label in [(0, COLOR_EMERALD, 'Pago regular (Y=0)'),
                            (1, COLOR_CARMIN, 'Default (Y=1)')]:
    subset = df_score[df_score['Y'] == clase]
    sns.kdeplot(subset['Score'], ax=axes[0], color=color, fill=True,
                alpha=0.45, label=label, linewidth=2)

axes[0].axvline(scores_test.median(), color=COLOR_INDIGO, linestyle='--',
                linewidth=1.5, alpha=0.7, label=f'Mediana: {scores_test.median():.0f}')
axes[0].set_xlabel('Score crediticio')
axes[0].set_ylabel('Densidad')
axes[0].set_title('Distribución del score por clase')
axes[0].legend()

# Panel derecho: boxplot
df_score_long = df_score.copy()
df_score_long['Y_label'] = df_score_long['Y'].map({0: 'Pago regular', 1: 'Default'})
sns.boxplot(data=df_score_long, x='Y_label', y='Score',
            palette=[COLOR_EMERALD, COLOR_CARMIN], ax=axes[1], width=0.5)
axes[1].set_xlabel('')
axes[1].set_ylabel('Score crediticio')
axes[1].set_title('Comparación por clase (boxplot)')

plt.tight_layout()
plt.show()


## 8. Métricas de evaluación: AUC, KS y Gini

Las tres métricas que la industria reporta junto a cualquier scorecard. Las calculamos sobre el **conjunto de testeo** (nunca sobre train).

| Métrica | Fórmula | Umbral aceptable |
|---------|---------|------------------|
| **AUC-ROC** | Área bajo la curva ROC | ≥ 0.70 |
| **KS** | $\max_s |F_{bad}(s) - F_{good}(s)|$ | ≥ 0.30 |
| **Gini** | $2 \cdot \text{AUC} - 1$ | ≥ 0.40 |


In [ ]:
# Cálculo de las tres métricas
auc = roc_auc_score(y_test, p_test)
gini = 2 * auc - 1

# KS de Kolmogorov-Smirnov entre las distribuciones del score por clase
scores_buenos = scores_test[y_test == 0]
scores_malos  = scores_test[y_test == 1]
ks_stat, ks_pvalue = stats.ks_2samp(scores_buenos, scores_malos)

# Interpretación según umbrales de la industria
def umbral_auc(v):
    if v >= 0.80: return '✅ Excelente'
    if v >= 0.75: return '✅ Bueno'
    if v >= 0.70: return '✅ Aceptable'
    if v >= 0.65: return '⚠️  Débil'
    return '❌ Insuficiente'

def umbral_ks(v):
    if v >= 0.50: return '✅ Excelente'
    if v >= 0.40: return '✅ Bueno'
    if v >= 0.30: return '✅ Aceptable'
    return '❌ Insuficiente'

def umbral_gini(v):
    if v >= 0.60: return '✅ Excelente'
    if v >= 0.50: return '✅ Bueno'
    if v >= 0.40: return '✅ Aceptable'
    return '❌ Insuficiente'

metricas = pd.DataFrame({
    'Métrica':         ['AUC-ROC', 'KS de Kolmogorov-Smirnov', 'Gini'],
    'Valor':           [auc, ks_stat, gini],
    'Umbral mínimo':   [0.70, 0.30, 0.40],
    'Evaluación':      [umbral_auc(auc), umbral_ks(ks_stat), umbral_gini(gini)]
})
metricas['Valor'] = metricas['Valor'].round(4)
print("📊 Métricas de performance sobre conjunto de testeo\n")
metricas


In [ ]:
# Visualización conjunta: curva ROC y curva KS
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# ═══ Panel izquierdo: Curva ROC ═══
fpr, tpr, _ = roc_curve(y_test, p_test)
axes[0].plot(fpr, tpr, color=COLOR_VIOLET, linewidth=2.5, label=f'AUC = {auc:.4f}')
axes[0].fill_between(fpr, tpr, alpha=0.15, color=COLOR_VIOLET)
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Modelo aleatorio')
axes[0].set_xlabel('Tasa de Falsos Positivos (FPR)')
axes[0].set_ylabel('Tasa de Verdaderos Positivos (TPR)')
axes[0].set_title('Curva ROC')
axes[0].legend(loc='lower right')
axes[0].grid(alpha=0.3)

# Anotación del Gini
axes[0].text(0.55, 0.20, f'Gini = 2·AUC − 1 = {gini:.4f}',
             fontsize=11, color=COLOR_INDIGO, fontweight='bold',
             bbox=dict(boxstyle='round,pad=0.5', facecolor='white',
                       edgecolor=COLOR_VIOLET))

# ═══ Panel derecho: Curva KS ═══
# Construcción de las CDF empíricas
score_grid = np.sort(scores_test.values)
cdf_good = np.array([(scores_buenos <= s).mean() for s in score_grid])
cdf_bad  = np.array([(scores_malos  <= s).mean() for s in score_grid])
diff = np.abs(cdf_bad - cdf_good)
ks_x = score_grid[np.argmax(diff)]

axes[1].plot(score_grid, cdf_good, color=COLOR_EMERALD, linewidth=2,
             label='CDF buenos (Y=0)')
axes[1].plot(score_grid, cdf_bad, color=COLOR_CARMIN, linewidth=2,
             label='CDF malos (Y=1)')
axes[1].vlines(ks_x, cdf_good[np.argmax(diff)], cdf_bad[np.argmax(diff)],
               color=COLOR_GOLD, linewidth=3, label=f'KS = {ks_stat:.4f}')

axes[1].set_xlabel('Score crediticio')
axes[1].set_ylabel('Función de distribución acumulada')
axes[1].set_title('Estadístico KS de Kolmogorov-Smirnov')
axes[1].legend(loc='lower right')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n🎯 Score óptimo de separación (KS máximo): {ks_x:.1f}")
print(f"   Este punto del score maximiza la distancia entre CDFs de buenos y malos")


### 8.1. Matriz de confusión con umbral óptimo

Aunque el modelo entrega probabilidades, en producción se aplica un umbral de decisión. Buscamos el **umbral óptimo según el criterio de Youden** ($J = \text{TPR} - \text{FPR}$).


In [ ]:
# Umbral óptimo por criterio de Youden
fpr, tpr, thresholds = roc_curve(y_test, p_test)
J = tpr - fpr
umbral_optimo = thresholds[np.argmax(J)]

# Predicciones binarias con el umbral óptimo
y_pred = (p_test >= umbral_optimo).astype(int)

# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm,
                     index=['Pago regular (real)', 'Default (real)'],
                     columns=['Pago regular (pred)', 'Default (pred)'])

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Purples', ax=ax,
            cbar_kws={'label': 'Frecuencia'}, linewidths=2, linecolor='white',
            annot_kws={'fontsize': 13, 'fontweight': 'bold'})
ax.set_title(f'Matriz de confusión\nUmbral óptimo (Youden): p ≥ {umbral_optimo:.3f}')
plt.tight_layout()
plt.show()

# Métricas derivadas
tn, fp, fn, tp = cm.ravel()
exactitud   = (tp + tn) / cm.sum()
sensibilidad = tp / (tp + fn) if (tp + fn) > 0 else 0
especificidad = tn / (tn + fp) if (tn + fp) > 0 else 0
precision = tp / (tp + fp) if (tp + fp) > 0 else 0

print(f"\n📊 Métricas con umbral óptimo (p ≥ {umbral_optimo:.3f}):")
print(f"   Exactitud (Accuracy)            : {exactitud:.4f}")
print(f"   Sensibilidad (Recall, TPR)      : {sensibilidad:.4f}")
print(f"   Especificidad (TNR)             : {especificidad:.4f}")
print(f"   Precisión (PPV)                 : {precision:.4f}")


## 9. Diagnóstico de supuestos

Antes de declarar válido el modelo, verificamos los **cuatro supuestos clave** de la regresión logística:

1. **Linealidad en los logits** (variables continuas)
2. **Multicolinealidad ausente o moderada** (VIF)
3. **Tamaño de muestra suficiente** (EPV, ya verificado en sección 4)
4. **Independencia de las observaciones** (no aplicable a este corte transversal)


### 9.1. Linealidad en los logits

Para cada variable continua, discretizamos en deciles, calculamos la **tasa de default empírica** por decil y la transformamos a logit. Si la relación con el centro del decil es aproximadamente lineal, el supuesto se cumple.


In [ ]:
def diagnostico_linealidad_logit(df_, variable, target='Y', n_bins=10):
    '''
    Verifica visualmente la linealidad de una variable continua en escala logit.
    '''
    df_local = df_[[variable, target]].copy()
    df_local['bin'] = pd.qcut(df_local[variable], q=n_bins, duplicates='drop')

    resumen = df_local.groupby('bin', observed=True).agg(
        centro=(variable, 'mean'),
        tasa_default=(target, 'mean'),
        n=(target, 'size')
    ).reset_index(drop=True)

    # Logit empírico (clip para evitar 0 y 1)
    resumen['p_clip'] = resumen['tasa_default'].clip(0.001, 0.999)
    resumen['logit_empirico'] = np.log(resumen['p_clip'] / (1 - resumen['p_clip']))

    return resumen


# Diagnóstico para cada variable continua
fig, axes = plt.subplots(1, len(vars_continuas), figsize=(4 * len(vars_continuas), 4))

for i, var in enumerate(vars_continuas):
    res = diagnostico_linealidad_logit(df, var)

    # Recta de regresión simple para referencia
    slope, intercept = np.polyfit(res['centro'], res['logit_empirico'], 1)
    x_line = np.linspace(res['centro'].min(), res['centro'].max(), 100)
    y_line = intercept + slope * x_line

    axes[i].scatter(res['centro'], res['logit_empirico'], s=80,
                    color=COLOR_VIOLET, edgecolor='white', linewidth=1.5, zorder=3)
    axes[i].plot(x_line, y_line, color=COLOR_GOLD, linewidth=2,
                 alpha=0.7, linestyle='--', label='Recta de referencia')
    axes[i].set_xlabel(var)
    axes[i].set_ylabel('Logit empírico')
    axes[i].set_title(var, fontsize=11)
    axes[i].grid(alpha=0.3)
    axes[i].legend(fontsize=8, loc='best')

plt.suptitle('Diagnóstico de linealidad en logits\n(puntos = deciles; recta = referencia lineal)',
             y=1.02, fontsize=12, color=COLOR_INDIGO, fontweight='bold')
plt.tight_layout()
plt.show()


### 📝 Lectura del diagnóstico

Si los puntos (deciles) **se alinean razonablemente con la recta de referencia dorada**, el supuesto de linealidad en logits se cumple. Si presentan curvatura sistemática, conviene:

- Aplicar transformaciones (logaritmo, raíz cuadrada, Box-Cox)
- O discretizar la variable con **WoE** (estándar industrial)

> 📌 En la práctica de *credit scoring* real, el binning con WoE es la solución preferida porque linealiza por construcción y maneja faltantes elegantemente.


### 9.2. Multicolinealidad: análisis VIF

El **Variance Inflation Factor** cuantifica cuánto incrementa la varianza de cada coeficiente debido a su correlación con las demás covariables. Umbrales:

- **VIF < 5**: multicolinealidad ausente o leve ✅
- **5 ≤ VIF < 10**: moderada ⚠️
- **VIF ≥ 10**: severa ❌


In [ ]:
# Cálculo del VIF sobre la matriz de diseño (sin intercepto para no inflar)
vif_data = pd.DataFrame()
vif_data['Variable'] = X_train_scaled.columns
vif_data['VIF'] = [variance_inflation_factor(X_train_scaled.values, i)
                   for i in range(X_train_scaled.shape[1])]
vif_data['VIF'] = vif_data['VIF'].round(2)


# Diagnóstico
def diagnostico_vif(v):
    if v < 5:   return '✅ Leve o ausente'
    if v < 10:  return '⚠️  Moderada'
    return '❌ Severa'


vif_data['Diagnóstico'] = vif_data['VIF'].apply(diagnostico_vif)
vif_data = vif_data.sort_values('VIF', ascending=False).reset_index(drop=True)

# Resumen
n_severa  = (vif_data['VIF'] >= 10).sum()
n_moderada = ((vif_data['VIF'] >= 5) & (vif_data['VIF'] < 10)).sum()

print(f"📊 Diagnóstico de multicolinealidad (VIF)\n")
print(f"   Variables con VIF severo (≥ 10):    {n_severa}")
print(f"   Variables con VIF moderado (5-10):  {n_moderada}")
print(f"   Variables sin problema (< 5):       {len(vif_data) - n_severa - n_moderada}\n")
vif_data


### 📝 Lectura del análisis VIF

- Si **todas las variables presentan VIF < 5**, no hay problema de multicolinealidad y el modelo es interpretable.
- Si aparecen valores moderados (5-10), conviene revisar cuáles variables están correlacionadas y considerar eliminar la menos informativa.
- VIF severos (≥ 10) indican redundancia explícita: dos o más variables aportan información casi idéntica.

> 💡 La codificación one-hot genera por construcción cierta colinearidad entre las dummies de una misma categoría, lo que puede inflar levemente los VIF de variables categóricas. Esto es esperable y no requiere intervención.


## 10. Síntesis y entregable de la Clase 5

### 🎯 Recapitulación del flujo construido

A lo largo de este notebook hemos:

1. **Cargado** el German Credit Dataset desde OpenML con fallback resiliente
2. **Recodificado** la variable respuesta según convención regulatoria (`bad → 1`)
3. **Preprocesado** las covariables: one-hot encoding, partición train/test y estandarización
4. **Verificado** el cumplimiento de la regla EPV de Peduzzi
5. **Ajustado** dos modelos en scikit-learn (con y sin regularización) y comparado sus coeficientes
6. **Ajustado** un modelo con statsmodels para obtener inferencia estadística completa
7. **Interpretado** los coeficientes mediante odds ratios e intervalos de confianza al 95 %
8. **Construido** una scorecard operativa con PDO = 20, Score_target = 600, Odds_target = 50:1
9. **Evaluado** la performance mediante AUC, KS y Gini
10. **Diagnosticado** los supuestos del modelo: linealidad en logits y multicolinealidad (VIF)

### 📋 Tabla resumen de resultados


In [ ]:
resumen_final = pd.DataFrame({
    'Métrica': [
        'Tamaño muestra train', 'Tamaño muestra test',
        'Variables modelo (con dummies)', 'EPV',
        'AUC-ROC (test)', 'KS (test)', 'Gini (test)',
        'Score min (test)', 'Score max (test)', 'Score mediana (test)'
    ],
    'Valor': [
        len(X_train_scaled), len(X_test_scaled),
        X_train_scaled.shape[1], f'{EPV:.1f}',
        f'{auc:.4f}', f'{ks_stat:.4f}', f'{gini:.4f}',
        f'{scores_test.min():.1f}', f'{scores_test.max():.1f}',
        f'{scores_test.median():.1f}'
    ]
})
print("📊 Resumen ejecutivo del modelo\n")
resumen_final


### 📋 Entregable parcial: Clase 6

Sobre la base de este notebook, ustedes deberán entregar (antes del inicio de la Clase 6) un notebook propio que incluya:

1. ✅ **Ajuste** de un modelo logístico con ≥ 5 variables explicativas (justificadas)
2. ✅ **Reporte** de los coeficientes con sus odds ratio e intervalos de confianza al 95 %
3. ✅ **Construcción** de la scorecard con PDO = 20, anclaje 600/50:1
4. ✅ **Cálculo** de AUC, KS y Gini sobre el conjunto de testeo
5. ✅ **Interpretación económica** fundamentada de tres coeficientes clave en lenguaje gerencial

#### 🎯 Criterios de evaluación

| Dimensión | Peso |
|-----------|------|
| Corrección técnica del código | 30 % |
| Interpretación económica de los coeficientes | 30 % |
| Construcción adecuada de la scorecard | 20 % |
| Calidad del reporte final | 20 % |

---

### 🔮 Anticipación: Clase 6 (jueves 14 de mayo)

En la próxima clase abandonamos el paradigma lineal y entramos al de las particiones recursivas:

- **Árboles de decisión CART**: criterios de partición (Gini, entropía), poda, visualización
- **Random Forest**: bagging, doble aleatorización, importancia de variables
- **Comparativa empírica** con el modelo logístico de hoy sobre el mismo dataset

> 💡 **Lectura sugerida**: capítulo 3.1 del apunte teórico de la Unidad 3, ya disponible en el repositorio del curso.

---

<center>

**Dr. Darío Ezequiel Díaz**
*Aplicaciones de Minería de Datos a Economía y Finanzas*
Maestría en Minería de Datos · UTN Facultad Regional Rosario · 2026

</center>
